In [0]:
# Top 10 Lifetime VIPs: Show the top 10 customers by total lifetime sales amount. Include their full name, marital status, gender, country, and age group (e.g., 18-30, 31-45).
query = """
WITH customer_stats AS (
    SELECT 
        c.customer_key,
        c.first_name || ' ' || c.last_name AS full_name,
        c.marital_status,
        c.gender,
        c.country,
        -- Calculate age based on birth_date
        FLOOR(DATEDIFF(CURRENT_DATE(), c.birth_date) / 365.25) AS age,
        SUM(f.sales_amount) AS total_lifetime_sales
    FROM bike_data.gold.fact_sales f
    JOIN bike_data.gold.dim_customer c ON f.customer_sk = c.customer_id
    GROUP BY 1, 2, 3, 4, 5, 6
)
SELECT 
    full_name,
    marital_status,
    gender,
    country,
    total_lifetime_sales,
    CASE 
        WHEN age < 30 THEN '18-30'
        WHEN age BETWEEN 31 AND 45 THEN '31-45'
        WHEN age BETWEEN 46 AND 60 THEN '46-60'
        ELSE '60+' 
    END AS age_group
FROM customer_stats
ORDER BY total_lifetime_sales DESC
LIMIT 10;
"""
df = spark.sql(query)
display(df)